<a href="https://colab.research.google.com/github/d3au-4/Pokemon-Predictor-Dominic-and-Nicholas/blob/main/Pokemon_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import os

In [19]:
#importing all of the kaggle datasets

sold_price_path = kagglehub.dataset_download("ryanheger/pokemon-card-sold-price-reference")

print("Path to dataset files:", path)

price_charting_path = kagglehub.dataset_download("zeynepcahan/pokemon-pricecharting")

print("Path to dataset files:", path)

print("Files in Dataset (pricecharting): ")

#############################################

for file in os.listdir(price_charting_path):
  print(file)

print("Files in Dataset (sold price): ")

for file in os.listdir(sold_price_path):
  print(file)

pricecharting_file = os.path.join(price_charting_path, "pokemon_pricecharting.csv")

sold_file = os.path.join(sold_price_path, "gemsnipe-card-sold-price-reference.csv")

price_df = pd.read_csv(pricecharting_file)
sold_df = pd.read_csv(sold_file)

Using Colab cache for faster access to the 'pokemon-card-sold-price-reference' dataset.
Path to dataset files: /root/.cache/kagglehub/datasets/zeynepcahan/pokemon-pricecharting/versions/1
Using Colab cache for faster access to the 'pokemon-pricecharting' dataset.
Path to dataset files: /root/.cache/kagglehub/datasets/zeynepcahan/pokemon-pricecharting/versions/1
Files in Dataset (pricecharting): 
pokemon_pricecharting.csv
Files in Dataset (sold price): 
gemsnipe-card-sold-price-reference.csv


Files in Dataset (pricecharting): 
pokemon_pricecharting.csv


In [ ]:
import requests
import json
import time

BASE_URL = "https://api.pokemontcg.io/v2"

def fetch_cards(query, page_size=20, max_pages=1):
    """Fetch cards matching a search query. No API key required for light use,
    but you'll hit rate limits faster without one — register free at
    https://dev.pokemontcg.io if you need to pull a lot of data."""
    all_cards = []
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "pageSize": page_size,
            "page": page,
        }
        response = requests.get(f"{BASE_URL}/cards", params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        all_cards.extend(data.get("data", []))

        # be polite to the free tier — small delay between pages
        time.sleep(0.5)

        if len(data.get("data", [])) < page_size:
            break  # no more pages

    return all_cards


def build_dataset(cards):
    """Flattens the raw API response into a clean dataset for your scoring pipeline."""
    rows = []
    for card in cards:
        tcgplayer_prices = card.get("tcgplayer", {}).get("prices", {})
        rows.append({
            "id": card.get("id"),
            "name": card.get("name"),
            "set_name": card.get("set", {}).get("name"),
            "set_release_date": card.get("set", {}).get("releaseDate"),
            "rarity": card.get("rarity"),
            "number": card.get("number"),
            "image_small": card.get("images", {}).get("small"),
            "image_large": card.get("images", {}).get("large"),
            # TCGplayer market price data IS included on pokemontcg.io cards
            # for free, even without the closed TCGplayer API — this is huge,
            # it means you don't need separate TCGplayer access at all.
            "market_price": tcgplayer_prices.get("holofoil", {}).get("market")
                            or tcgplayer_prices.get("normal", {}).get("market"),
            "low_price": tcgplayer_prices.get("holofoil", {}).get("low")
                         or tcgplayer_prices.get("normal", {}).get("low"),
            "high_price": tcgplayer_prices.get("holofoil", {}).get("high")
                          or tcgplayer_prices.get("normal", {}).get("high"),
        })
    return rows


if __name__ == "__main__":
    print("Fetching Charizard cards from pokemontcg.io...")
    cards = fetch_cards("name:charizard", page_size=20, max_pages=1)
    print(f"Retrieved {len(cards)} cards")

    dataset = build_dataset(cards)

    # Save to JSON so you can load it in your scoring pipeline
    with open("charizard_dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)

    print("\nSample results:")
    for row in dataset[:5]:
        print(f"  {row['name']} ({row['set_name']}, {row['rarity']}) — market: ${row['market_price']}")

    print(f"\nSaved {len(dataset)} cards to charizard_dataset.json")


Fetching Charizard cards from pokemontcg.io...


HTTPError: 500 Server Error: Internal Server Error for url: https://api.pokemontcg.io/v2/cards?q=name%3Acharizard&pageSize=20&page=1